# NV-Embed Item Embedding Extraction

This notebook extracts dense item embeddings using [NV-Embed-v2](https://huggingface.co/nvidia/NV-Embed-v2) (`nvidia/NV-Embed-v2`).

**Run this notebook before** `1_hierarchical_semantics_indexing.ipynb`. The embeddings are saved to disk and loaded automatically by the indexing notebook.

**Output:** `{embedding_dir}/{dataset}_nvembed_len{max_length}.npy`

**Requirements:**
```bash
pip install torch>=2.0.0 transformers>=4.36.0 numpy tqdm
```

> **Note:** NV-Embed-v2 requires `torch >= 2.0.0` and `transformers >= 4.36.0`.  
> These versions may be **incompatible with the GRAM training environment**.  
> We recommend running this notebook in a separate environment before training.

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ===== Dataset Configuration =====
dataset_name = "Beauty"          # Options: Beauty, Sports, Toys, Yelp
data_path = "../rec_datasets"    # Path to rec_datasets directory
embedding_dir = "./embeddings"   # Directory to save item embeddings
os.makedirs(embedding_dir, exist_ok=True)

# ===== NV-Embed Hyperparameters =====
max_length = 4096   # Maximum input sequence length (use 128 for quick testing)
batch_size = 4      # Batch size for encoding (reduce if OOM)

print(f"Dataset: {dataset_name}, max_length: {max_length}, batch_size: {batch_size}")

## Step 1: Load Item Data

In [ ]:
def read_lines(path):
    with open(path, 'r') as f:
        return [line.rstrip('\n') for line in f]


def get_dict_from_lines(lines):
    index_map = {}
    for line in lines:
        info = line.split(' ', 1)
        if len(info) == 2:
            index_map[info[0]] = info[1]
    return index_map


item_text_file = os.path.join(data_path, dataset_name, "item_plain_text.txt")
item2input = get_dict_from_lines(read_lines(item_text_file))
item_id_list = list(item2input.keys())
item_text_list = list(item2input.values())

print(f"Loaded {len(item2input)} items from {dataset_name}")
print(f"Sample: {item_id_list[0]}")
print(f"  Text: {item_text_list[0][:150]}...")

## Step 2: Load NV-Embed Model

In [ ]:
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading NV-Embed-v2 on {device}...")
nvembed_model = AutoModel.from_pretrained('nvidia/NV-Embed-v2', trust_remote_code=True)
nvembed_model = nvembed_model.to(device)
nvembed_model.eval()
print("Model loaded.")

## Step 3: Extract and Save Embeddings

Embeddings are cached to disk. If the output file already exists, this step is skipped.

In [ ]:
embedding_file = os.path.join(embedding_dir, f"{dataset_name.lower()}_nvembed_len{max_length}.npy")

if os.path.exists(embedding_file):
    print(f"Embeddings already exist at {embedding_file}. Skipping extraction.")
    item_embeddings = np.load(embedding_file)
else:
    print(f"Extracting embeddings for {len(item_text_list)} items...")
    embeddings_list = []

    with torch.no_grad():
        for i in tqdm(range(0, len(item_text_list), batch_size), desc="Encoding"):
            batch_texts = item_text_list[i:i + batch_size]
            embeddings = nvembed_model.encode(batch_texts, instruction="", max_length=max_length)
            embeddings = F.normalize(embeddings, p=2, dim=1)
            embeddings_list.append(embeddings.cpu().numpy())

    item_embeddings = np.vstack(embeddings_list)
    np.save(embedding_file, item_embeddings)
    print(f"Saved embeddings to {embedding_file}")

print(f"Embedding shape: {item_embeddings.shape}")